In [3]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load dataset
data = pd.read_csv('HotelCustomersDataset.csv', sep='\t')

# Display column names
#print(data.columns)

# Handle missing values
data['Age'].fillna(data['Age'].mean(), inplace=True)

# Encode categorical variables
label_encoder = LabelEncoder() # Fit and transform the Nationality column
data['Nationality'] = label_encoder.fit_transform(data['Nationality'])
data['DistributionChannel'] = label_encoder.fit_transform(data['DistributionChannel'])
data['MarketSegment'] = label_encoder.fit_transform(data['MarketSegment'])

# List of columns to combine
columns_to_combine = ['SRHighFloor', 'SRLowFloor', 'SRAccessibleRoom', 'SRMediumFloor',
                      'SRBathtub', 'SRShower', 'SRCrib', 'SRKingSizeBed', 'SRTwinBed',
                      'SRNearElevator', 'SRAwayFromElevator', 'SRNoAlcoholInMiniBar', 'SRQuietRoom']

# Create a new column that combines the specified columns
data['SpecialRequests'] = data[columns_to_combine].sum(axis=1)

# Remove the original columns
data.drop(columns=columns_to_combine, inplace=True)

# Display the updated dataset to verify the changes
print("Updated Dataset:")
print(data.head())

# Define features (X) and target label (y)
# Here assuming 'BookingsCheckedIn' as the target label for classification
# Replace 'BookingsCheckedIn' with your actual target column if different
X = data.drop(columns=['ID', 'NameHash', 'DocIDHash', 'BookingsCheckedIn'])
y = data['BookingsCheckedIn']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


Updated Dataset:
   ID  Nationality        Age  DaysSinceCreation  \
0   1          137  51.000000                150   
1   2          137  45.398028               1095   
2   3           43  31.000000               1095   
3   4           57  60.000000               1095   
4   5           57  51.000000               1095   

                                            NameHash  \
0  0x8E0A7AF39B633D5EA25C3B7EF4DFC5464B36DB7AF375...   
1  0x21EDE41906B45079E75385B5AA33287CA09DE1AB86DE...   
2  0x31C5E4B74E23231295FDB724AD578C02C4A723F4BA2B...   
3  0xFF534C83C0EF23D1CE516BC80A65D0197003D27937D4...   
4  0x9C1DEF02C9BE242842C1C1ABF2C5AA249A1EEB4763B4...   

                                           DocIDHash  AverageLeadTime  \
0  0x71568459B729F7A7ABBED6C781A84CA4274D571003AC...               45   
1  0x5FA1E0098A31497057C5A6B9FE9D49FD6DD47CCE7C26...               61   
2  0xC7CF344F5B03295037595B1337AC905CA188F1B5B3A5...                0   
3  0xBD3823A9B4EC35D6CAF4B27AE423A677C020

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Initialize the model
model = RandomForestClassifier(random_state=42)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)



In [5]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(conf_matrix)

# Classification report
class_report = classification_report(y_test, y_pred, zero_division=0)
print('Classification Report:')
print(class_report)


Accuracy: 0.99
Confusion Matrix:
[[ 3923     0     0     0     0     0     0     0     0     0     0     0
      0]
 [    0 12527     1     0     0     0     0     0     0     0     0     0
      0]
 [    0   151    64     2     0     0     0     0     0     0     0     0
      0]
 [    0    16     9     1     1     0     0     0     0     0     0     0
      0]
 [    0     4     2     0     1     0     0     0     0     0     0     0
      0]
 [    0     0     1     0     1     0     0     0     1     0     0     0
      0]
 [    0     1     1     0     0     0     0     0     0     0     0     0
      0]
 [    0     2     2     0     0     0     1     0     0     0     0     1
      0]
 [    0     0     0     0     0     0     0     0     0     0     0     0
      0]
 [    0     0     0     0     0     0     0     2     0     0     0     0
      0]
 [    0     0     0     0     0     0     0     0     0     0     0     0
      1]
 [    0     0     0     0     1     0     1     0     

In [6]:
# Feature importance
importances = model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
print('Feature Importances:')
print(feature_importance_df.head())


Feature Importances:
               Feature  Importance
10   DaysSinceLastStay    0.209844
9           RoomNights    0.181301
11  DaysSinceFirstStay    0.148496
5         OtherRevenue    0.138328
8        PersonsNights    0.132954


In [7]:
feature_importance_df['Importance'] = feature_importance_df['Importance'].round(3)
feature_importance_df

,Feature,Importance
10,DaysSinceLastStay,0.210
9,RoomNights,0.181
11,DaysSinceFirstStay,0.148
5,OtherRevenue,0.138
8,PersonsNights,0.133
4,LodgingRevenue,0.073
3,AverageLeadTime,0.070
2,DaysSinceCreation,0.020
1,Age,0.009
13,MarketSegment,0.005
